# Experiment 10 — Theme Business Needs Batch

Theme business needs + each Epic's Stage(s) + base L3 candidates, with all preflight-valid Epics in a Theme classified together in one LLM call.

## What the LLM sees

One LLM call is made per Theme for all preflight-valid Epics under that Theme.

```text
task
theme
  └─ business_needs
epics[]
  ├─ epic_key
  └─ stages[]
      ├─ value_stream_stage
      │   ├─ stage_id
      │   ├─ stage_name
      │   ├─ stage_description
      │   ├─ entrance_criteria
      │   └─ exit_criteria
      └─ candidate_l3_capabilities[]
          ├─ capability_id
          ├─ capability_name
          ├─ capability_description
          └─ capability_tier
selection_instruction
```

**Not sent to the LLM:** Theme description, Epic description, Epic success criteria, L1/L2 hierarchy, ground truth.


## Configuration and imports

In [ ]:
from pathlib import Path
from time import perf_counter
import ast
import io
import json
import os
import re
import tokenize

import httpx
import pandas as pd
from IPython.display import display

from common import (
    call_llm_with_metrics,
    load_gateway,
    parse_json_response,
    save_results_excel,
    score_sets,
    validate_l3_response,
)

NOTEBOOK_DIR = Path.cwd()
WORKSPACE_DIR = (
    NOTEBOOK_DIR.parent
    if (NOTEBOOK_DIR.parent / "epic_gen.csv").exists()
    else NOTEBOOK_DIR
)
DATA_DIR = Path(os.getenv("L3_EXPERIMENT_DATA_DIR", WORKSPACE_DIR))

THEME_PATH = Path(os.getenv("L3_THEME_PATH", DATA_DIR / "epic_gen.csv"))
STAGE_PATH = Path(os.getenv("L3_STAGE_PATH", DATA_DIR / "VSSrv.csv"))
STAGE_CAPABILITY_MAP_PATH = Path(
    os.getenv(
        "L3_STAGE_CAPABILITY_MAP_PATH",
        DATA_DIR / "VSSCaprv (1).csv",
    )
)
GROUND_TRUTH_PATH = Path(
    os.getenv(
        "L3_GROUND_TRUTH_PATH",
        NOTEBOOK_DIR / "epic_l3_ground_truth_all_themes.xlsx",
    )
)

# Match the current comparison population: first 20 Themes. Ground truth is not consulted here.
THEME_IDS = (
    pd.read_csv(
        THEME_PATH,
        usecols=["key"],
        encoding="cp1252",
        encoding_errors="replace",
        dtype=str,
    )["key"]
    .dropna()
    .str.strip()
    .loc[lambda values: values.ne("")]
    .drop_duplicates()
    .head(20)
    .tolist()
)

VALUE_STREAM_STAGE_FIELD_ID = "customfield_18700"

# Optional single-example inspection. Leave None for batch execution only.
INSPECTION_THEME_ID = None
INSPECTION_EPIC_KEY = None

EXPERIMENT_NAME = "E10_BUSINESS_NEEDS_THEME_BATCH"


## Retrieval

In [ ]:
def clean_text(v):
    if v is None or (not isinstance(v, (list, dict)) and pd.isna(v)):
        return ""
    return str(v).strip()

def parse_exported_list(v):
    if v is None or pd.isna(v):
        return []
    text = str(v).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        values = [ast.literal_eval(t.string) for t in tokenize.generate_tokens(io.StringIO(text).readline) if t.type == tokenize.STRING]
        if values:
            return [clean_text(x) for x in values]
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [text]
    return [clean_text(x) for x in parsed] if isinstance(parsed, (list, tuple, set)) else [clean_text(parsed)]

def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, dtype=str, encoding="cp1252", encoding_errors="replace")
    return pd.read_excel(path, dtype=str)

def load_themes():
    frame = read_table(THEME_PATH)
    frame = frame.loc[frame["key"].isin(THEME_IDS)]
    themes = {}

    for _, row in frame.iterrows():
        epic_keys = parse_exported_list(row.get("epic_keys"))
        themes[clean_text(row["key"])] = {
            "theme_description": clean_text(row.get("description")),
            "theme_business_needs": clean_text(row.get("businessNeeds")),
            "epics": [{"key": epic_key} for epic_key in epic_keys],
        }

    return themes

def jira_headers():
    if os.getenv("JIRA_HEADERS_JSON"):
        return json.loads(os.environ["JIRA_HEADERS_JSON"])
    token = os.getenv("JIRA_BEARER_TOKEN") or os.getenv("JIRA_TOKEN")
    if token:
        return {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    raise RuntimeError("Set JIRA_HEADERS_JSON, JIRA_BEARER_TOKEN, or JIRA_TOKEN.")

def epic_stage_ids(epic_key):
    url = f"{os.environ['JIRA_BASE_URL'].rstrip('/')}/rest/api/2/issue/{epic_key}"
    verify = os.getenv("JIRA_VERIFY_SSL", "false").lower() == "true"
    with httpx.Client(headers=jira_headers(), verify=verify, timeout=30) as client:
        response = client.get(url, params={"fields": VALUE_STREAM_STAGE_FIELD_ID})
        response.raise_for_status()
    raw = response.json().get("fields", {}).get(VALUE_STREAM_STAGE_FIELD_ID) or []
    raw = raw if isinstance(raw, list) else [raw]
    ids = []
    for value in raw:
        text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else clean_text(value)
        ids.extend(re.findall(r"VSS\d+", text, flags=re.IGNORECASE))
    return list(dict.fromkeys(x.upper() for x in ids))

themes = load_themes()
stage_frame = read_table(STAGE_PATH)
stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)

def stage_context(stage_id):
    match = stage_frame.loc[stage_frame["Value Stream Stage ID"].astype(str).str.strip() == stage_id]
    if match.empty:
        raise KeyError(f"No stage metadata for {stage_id}")
    row = match.iloc[0]
    return {
        "stage_id": stage_id,
        "stage_name": clean_text(row["Value Stream Stage Name"]),
        "stage_description": clean_text(row["Value Stream Stage Description"]),
        "entrance_criteria": clean_text(row["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean_text(row["Value Stream Stage Exit Criteria"]),
    }


## Candidate construction

In [ ]:
def candidate_rows_for_stage(stage_id):
    rows = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"].astype(str).str.strip()
        == stage_id
    ].copy()
    rows = (
        rows.drop_duplicates(subset=["Capability ID"], keep="first")
        .sort_values(["Capability Name", "Capability ID"], kind="stable")
    )

    return [
        {
            "capability_id": clean_text(row["Capability ID"]),
            "capability_name": clean_text(row["Capability Name"]),
            "capability_description": clean_text(row["Capability Description"]),
            "capability_tier": clean_text(row["Capability Tier"]),
        }
        for _, row in rows.iterrows()
    ]


## Production prompt

In [ ]:
SYSTEM_PROMPT = """You are an enterprise Business Capability Architecture specialist performing Level 3 (L3) business capability classification.

OBJECTIVE
For each Epic in the supplied Theme batch, select only candidate L3 capabilities materially represented by that Epic's supplied Value Stream Stage context and the shared Theme context. Classify each Epic independently. This is capability classification, not keyword matching.

EVIDENCE PRIORITY
Use only fields that are present, in this order:
1. Value Stream Stage context for that Epic
2. Theme business needs
3. Theme description

Theme description, Epic description, and Epic success criteria are not supplied. Do not assume them.

CANDIDATE INTERPRETATION
- capability_description is the primary semantic definition.
- capability_name is the supporting label.
- capability_tier is supporting taxonomy context only.
- level_1_name and level_2_name, when supplied, are for disambiguation only and must never independently justify a selection.

DECISION PROCEDURE
1. For each Epic, determine the business function or outcome supported by its supplied Stage context and the shared Theme context.
2. Compare that evidence semantically against that Epic's candidates.
3. Select a candidate only when the supplied evidence materially supports its business function; relatedness alone is insufficient.
4. Stage membership alone is not enough; use Stage meaning together with Theme context.
5. When candidates overlap, prefer the most specific directly aligned capability.

EPIC ISOLATION
- Use only candidates listed under the Epic being classified.
- Do not use another Epic's Stage or candidates as evidence for the current Epic.
- Shared Theme membership does not mean different Epics should receive the same L3 selection.

DO NOT SELECT
Do not select a capability merely because of shared keywords, hierarchy family, Stage membership, upstream/downstream relationship, data exchange, stakeholder involvement, technical adjacency, or general Theme relevance.

MULTI-SELECTION
Select 0 to 3 capabilities per Epic. Default to one when one capability adequately represents the function. Select multiple only for distinct material business functions with independent evidence. Return an empty l3 list when none is sufficiently supported.

REASONS
For every selection, give a concise reason connecting supplied evidence to the candidate definition. Use only exact capability_id values from that Epic's supplied candidates; never invent or alter an ID.

FINAL VALIDATION
Before responding, verify that every supplied epic_key appears exactly once, every selected ID belongs to that Epic's candidates, every selection has direct evidence, no selection is merely adjacent, and no more than three capabilities are selected per Epic.

OUTPUT CONTRACT
Return JSON only, with no Markdown, code fences, commentary, or extra fields:
{"epics":[{"epic_key":"GROUP-00000","l3":[{"capability_id":"CAP00000000","reason":"Concise evidence-based explanation."}]}]}"""


def build_theme_batch_prompt(theme, epic_payloads):
    payload = {
        "task": "Classify every supplied Epic independently and select 0 to 3 L3 capabilities for each.",
        "theme": {
            "business_needs": theme["theme_business_needs"],
        },
        "epics": epic_payloads,
        "selection_instruction": "Return one result for every epic_key. Use only candidates listed under that Epic.",
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


## Prediction

In [ ]:
def build_batch_epics(theme_rows):
    epic_payloads = []
    allowed_ids_by_epic = {}

    for row in theme_rows.to_dict(orient="records"):
        epic_key = row["epic_key"]
        stages = []
        allowed_ids = set()

        for stage_id in json.loads(row["stage_ids"]):
            candidates = candidate_rows_for_stage(stage_id)
            stages.append({
                "value_stream_stage": stage_context(stage_id),
                "candidate_l3_capabilities": candidates,
            })
            allowed_ids.update(
                candidate["capability_id"] for candidate in candidates
            )

        epic_payloads.append({
            "epic_key": epic_key,
            "stages": stages,
        })
        allowed_ids_by_epic[epic_key] = sorted(allowed_ids)

    return epic_payloads, allowed_ids_by_epic


def validate_theme_batch_response(
    payload,
    expected_epic_keys,
    allowed_ids_by_epic,
):
    if not isinstance(payload, dict) or set(payload) != {"epics"}:
        raise ValueError("Theme-batch response must contain only the 'epics' field")
    if not isinstance(payload["epics"], list):
        raise ValueError("Theme-batch 'epics' must be a list")

    expected_epic_keys = list(expected_epic_keys)
    expected_set = set(expected_epic_keys)
    seen = {}

    for item in payload["epics"]:
        if not isinstance(item, dict):
            raise ValueError("Each Epic result must be an object")
        epic_key = str(item.get("epic_key", "")).strip()
        if epic_key not in expected_set:
            raise ValueError(f"Unexpected epic_key in response: {epic_key}")
        if epic_key in seen:
            raise ValueError(f"Duplicate epic_key in response: {epic_key}")

        selections = validate_l3_response(
            {"l3": item.get("l3", [])},
            allowed_ids_by_epic[epic_key],
            allow_empty=True,
            max_selected=3,
        )
        seen[epic_key] = selections

    missing = [key for key in expected_epic_keys if key not in seen]
    if missing:
        raise ValueError(f"Missing Epic results: {missing}")

    return seen


def predict_theme_batch(
    gateway,
    theme,
    epic_payloads,
    allowed_ids_by_epic,
):
    user_prompt = build_theme_batch_prompt(theme, epic_payloads)
    raw_response, metrics = call_llm_with_metrics(
        gateway,
        SYSTEM_PROMPT,
        user_prompt,
    )
    expected_epic_keys = [item["epic_key"] for item in epic_payloads]
    predictions = validate_theme_batch_response(
        parse_json_response(raw_response),
        expected_epic_keys,
        allowed_ids_by_epic,
    )
    return {
        "user_prompt": user_prompt,
        "raw_response": raw_response,
        "predictions": predictions,
        "metrics": metrics,
    }


def metric_text(value):
    return "n/a" if value is None else str(value)


def summarize_llm_calls(call_metrics):
    successful = call_metrics.loc[call_metrics["status"] == "ok"].copy()

    def numeric(column):
        return pd.to_numeric(successful[column], errors="coerce").dropna()

    latency = numeric("latency_seconds")
    input_tokens = numeric("input_tokens")
    output_tokens = numeric("output_tokens")
    total_tokens = numeric("total_tokens")
    epics_in_call = numeric("epics_in_call")
    total_epics_in_calls = int(epics_in_call.sum()) if len(epics_in_call) else 0
    total_token_count = int(total_tokens.sum()) if len(total_tokens) else None

    return pd.DataFrame([{
        "successful_calls": len(successful),
        "failed_calls": int((call_metrics["status"] == "error").sum()),
        "usage_reported_calls": len(total_tokens),
        "total_epics_in_calls": total_epics_in_calls,
        "avg_epics_per_call": float(epics_in_call.mean()) if len(epics_in_call) else None,
        "avg_latency_seconds": float(latency.mean()) if len(latency) else None,
        "p50_latency_seconds": float(latency.quantile(0.50)) if len(latency) else None,
        "p95_latency_seconds": float(latency.quantile(0.95)) if len(latency) else None,
        "avg_input_tokens": float(input_tokens.mean()) if len(input_tokens) else None,
        "avg_output_tokens": float(output_tokens.mean()) if len(output_tokens) else None,
        "avg_total_tokens": float(total_tokens.mean()) if len(total_tokens) else None,
        "total_input_tokens": int(input_tokens.sum()) if len(input_tokens) else None,
        "total_output_tokens": int(output_tokens.sum()) if len(output_tokens) else None,
        "total_tokens": total_token_count,
        "tokens_per_epic": (
            float(total_token_count / total_epics_in_calls)
            if total_token_count is not None and total_epics_in_calls
            else None
        ),
    }])


def run_predictions(preflight):
    eligible_rows = preflight.loc[preflight["evaluation_eligible"]].copy()
    prediction_rows = []
    call_rows = []

    call_columns = [
        "experiment",
        "theme_id",
        "epics_in_call",
        "stage_count",
        "candidate_instances",
        "status",
        "latency_seconds",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "selected_count",
        "error",
    ]

    theme_groups = list(eligible_rows.groupby("theme_id", sort=False))
    print(
        f"\nRunning {EXPERIMENT_NAME}: "
        f"{len(eligible_rows)} valid Epics in {len(theme_groups)} Theme-level calls"
    )

    if eligible_rows.empty:
        return pd.DataFrame(), pd.DataFrame(columns=call_columns)

    gateway = load_gateway()

    for theme_index, (theme_id, theme_rows) in enumerate(theme_groups, start=1):
        theme = themes[theme_id]
        epic_payloads, allowed_ids_by_epic = build_batch_epics(theme_rows)
        stage_count = sum(len(item["stages"]) for item in epic_payloads)
        candidate_instances = sum(
            len(stage["candidate_l3_capabilities"])
            for item in epic_payloads
            for stage in item["stages"]
        )
        started = perf_counter()

        print(
            f"\n[THEME LLM {theme_index}/{len(theme_groups)}] {theme_id}"
            f" | epics={len(epic_payloads)}"
            f" | stages={stage_count}"
            f" | candidate_instances={candidate_instances}"
        )

        try:
            result = predict_theme_batch(
                gateway,
                theme,
                epic_payloads,
                allowed_ids_by_epic,
            )
            metrics = result["metrics"]
            predictions_by_epic = result["predictions"]
            selected_count = sum(
                len(selections) for selections in predictions_by_epic.values()
            )

            print(
                f"  OK"
                f" | latency={metrics['latency_seconds']:.3f}s"
                f" | input_tokens={metric_text(metrics['input_tokens'])}"
                f" | output_tokens={metric_text(metrics['output_tokens'])}"
                f" | total_tokens={metric_text(metrics['total_tokens'])}"
                f" | selected={selected_count}"
            )

            call_rows.append({
                "experiment": EXPERIMENT_NAME,
                "theme_id": theme_id,
                "epics_in_call": len(epic_payloads),
                "stage_count": stage_count,
                "candidate_instances": candidate_instances,
                "status": "ok",
                "latency_seconds": metrics["latency_seconds"],
                "input_tokens": metrics["input_tokens"],
                "output_tokens": metrics["output_tokens"],
                "total_tokens": metrics["total_tokens"],
                "selected_count": selected_count,
                "error": None,
            })

            for row in theme_rows.to_dict(orient="records"):
                epic_key = row["epic_key"]
                selections = predictions_by_epic[epic_key]
                predicted_ids = sorted({
                    selection["capability_id"] for selection in selections
                })
                prediction_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": epic_key,
                    "stage_ids": row["stage_ids"],
                    "ground_truth_l3_ids": row["ground_truth_l3_ids"],
                    "available_candidate_l3_ids": row["available_candidate_l3_ids"],
                    "gt_found_in_candidates": row["gt_found_in_candidates"],
                    "gt_missing_from_candidates": row["gt_missing_from_candidates"],
                    "predicted_l3_ids": json.dumps(predicted_ids),
                    "model_reasons": json.dumps(selections, ensure_ascii=False),
                    "status": "ok",
                    "error": None,
                })
        except Exception as exc:
            latency = perf_counter() - started
            error = str(exc)
            print(f"  ERROR | latency={latency:.3f}s | {error}")
            call_rows.append({
                "experiment": EXPERIMENT_NAME,
                "theme_id": theme_id,
                "epics_in_call": len(epic_payloads),
                "stage_count": stage_count,
                "candidate_instances": candidate_instances,
                "status": "error",
                "latency_seconds": latency,
                "input_tokens": None,
                "output_tokens": None,
                "total_tokens": None,
                "selected_count": None,
                "error": error,
            })
            for row in theme_rows.to_dict(orient="records"):
                prediction_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": row["epic_key"],
                    "stage_ids": row["stage_ids"],
                    "ground_truth_l3_ids": row["ground_truth_l3_ids"],
                    "available_candidate_l3_ids": row["available_candidate_l3_ids"],
                    "gt_found_in_candidates": row["gt_found_in_candidates"],
                    "gt_missing_from_candidates": row["gt_missing_from_candidates"],
                    "predicted_l3_ids": json.dumps([]),
                    "model_reasons": json.dumps([]),
                    "status": "error",
                    "error": error,
                })

    return (
        pd.DataFrame(prediction_rows),
        pd.DataFrame(call_rows, columns=call_columns),
    )


## Batch preflight, Theme-batch execution, and evaluation

Ground truth is loaded **before any LLM call** only to validate the experiment population. Only preflight-valid Epics are included in a Theme batch. GT is never included in the model prompt.


In [ ]:
def ground_truth_by_epic():
    gt = read_table(GROUND_TRUTH_PATH).copy()
    gt["l3_capability_id"] = (
        gt["l3_capability_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )
    gt = gt.loc[gt["l3_capability_id"].ne("")]
    return {
        key: set(group["l3_capability_id"])
        for key, group in gt.groupby("epic_key", sort=False)
    }


def preflight_population():
    truth = ground_truth_by_epic()
    rows = []
    total_epics = sum(len(theme["epics"]) for theme in themes.values())
    check_index = 0

    print("\n================ PRECHECK ================")
    print(f"Themes selected: {len(themes)}")
    print(f"Total Epics: {total_epics}")

    for theme_id, theme in themes.items():
        for epic in theme["epics"]:
            check_index += 1
            epic_key = epic["key"]
            gt_ids = truth.get(epic_key)
            stage_ids = []
            candidate_ids = set()
            reason = ""
            error = None

            print(
                f"\n[GT CHECK {check_index}/{total_epics}] "
                f"{theme_id} | {epic_key}"
            )

            if gt_ids is None:
                reason = "missing_ground_truth"
            else:
                try:
                    stage_ids = epic_stage_ids(epic_key)
                except Exception as exc:
                    reason = "error"
                    error = str(exc)

                if not reason and not stage_ids:
                    reason = "no_stage"

                if not reason:
                    try:
                        for stage_id in stage_ids:
                            # Validate stage metadata now so invalid rows never
                            # reach the LLM phase.
                            stage_context(stage_id)
                            candidates = candidate_rows_for_stage(stage_id)
                            candidate_ids.update(
                                candidate["capability_id"]
                                for candidate in candidates
                            )
                    except Exception as exc:
                        reason = "error"
                        error = str(exc)

                if not reason and not candidate_ids:
                    reason = "no_candidates"

            if gt_ids is None:
                gt_found = set()
                gt_missing = set()
            else:
                gt_found = gt_ids & candidate_ids
                gt_missing = gt_ids - candidate_ids

            if not reason and gt_missing:
                reason = "gt_not_fully_retrievable"

            eligible = gt_ids is not None and not reason

            print(f"Stages: {stage_ids}")
            print(f"GT L3s: {sorted(gt_ids) if gt_ids is not None else []}")
            print(f"Candidate L3s: {sorted(candidate_ids)}")
            print(f"GT found in candidates: {sorted(gt_found)}")
            print(f"GT missing from candidates: {sorted(gt_missing)}")

            if eligible:
                print("STATUS: VALID")
            else:
                detail = f" | {error}" if error else ""
                print(f"STATUS: INVALID - {reason}{detail}")

            rows.append({
                "experiment": EXPERIMENT_NAME,
                "theme_id": theme_id,
                "epic_key": epic_key,
                "stage_ids": json.dumps(stage_ids),
                "ground_truth_l3_ids": (
                    json.dumps(sorted(gt_ids))
                    if gt_ids is not None
                    else None
                ),
                "available_candidate_l3_ids": json.dumps(
                    sorted(candidate_ids)
                ),
                "gt_found_in_candidates": json.dumps(sorted(gt_found)),
                "gt_missing_from_candidates": json.dumps(
                    sorted(gt_missing)
                ),
                "evaluation_eligible": eligible,
                "evaluation_exclusion_reason": reason,
                "preflight_error": error,
            })

    preflight = pd.DataFrame(rows)
    valid_count = int(preflight["evaluation_eligible"].sum())
    invalid_count = len(preflight) - valid_count

    print("\n================ PRECHECK SUMMARY ================")
    print(f"Themes selected: {len(themes)}")
    print(f"Total Epics: {len(preflight)}")
    print(f"VALID Epics: {valid_count}")
    print(f"INVALID Epics: {invalid_count}")
    print("Invalid breakdown:")
    for reason in (
        "missing_ground_truth",
        "no_stage",
        "no_candidates",
        "gt_not_fully_retrievable",
        "error",
    ):
        count = int(
            (preflight["evaluation_exclusion_reason"] == reason).sum()
        )
        print(f"  {reason}: {count}")
    valid_theme_count = int(preflight.loc[preflight["evaluation_eligible"], "theme_id"].nunique())
    print(f"Valid Epics available for batching: {valid_count}")
    print(f"Theme-level LLM calls planned: {valid_theme_count}")
    print("==================================================")

    return preflight


def evaluate_predictions(prediction_frame):
    out = []
    for row in prediction_frame.to_dict(orient="records"):
        pred = set(json.loads(row["predicted_l3_ids"]))
        gt = set(json.loads(row["ground_truth_l3_ids"]))

        if row["status"] == "error":
            metrics = {
                "exact_match": None,
                "precision": None,
                "recall": None,
                "f1": None,
                "predicted_count": len(pred),
                "truth_count": len(gt),
            }
        else:
            metrics = score_sets(pred, gt)

        row.update(metrics)
        out.append(row)

    return pd.DataFrame(out)


def evaluation_summary(results, preflight):
    scored = results.loc[results["exact_match"].notna()]
    valid_preflight = int(preflight["evaluation_eligible"].sum())

    summary = pd.DataFrame([{
        "scope": "valid_evaluation_population",
        "evaluated_epics": len(scored),
        "exact_match_accuracy": (
            scored["exact_match"].mean() if len(scored) else 0.0
        ),
        "mean_precision": (
            scored["precision"].mean() if len(scored) else 0.0
        ),
        "mean_recall": (
            scored["recall"].mean() if len(scored) else 0.0
        ),
        "mean_f1": scored["f1"].mean() if len(scored) else 0.0,
    }])

    diagnostics = pd.DataFrame([{
        "themes_selected": len(themes),
        "total_epics": len(preflight),
        "preflight_valid_epics": valid_preflight,
        "preflight_invalid_epics": len(preflight) - valid_preflight,
        "missing_ground_truth": int((
            preflight["evaluation_exclusion_reason"]
            == "missing_ground_truth"
        ).sum()),
        "no_stage": int((
            preflight["evaluation_exclusion_reason"] == "no_stage"
        ).sum()),
        "no_candidates": int((
            preflight["evaluation_exclusion_reason"] == "no_candidates"
        ).sum()),
        "gt_not_fully_retrievable": int((
            preflight["evaluation_exclusion_reason"]
            == "gt_not_fully_retrievable"
        ).sum()),
        "preflight_errors": int((
            preflight["evaluation_exclusion_reason"] == "error"
        ).sum()),
        "llm_prediction_errors": int((
            results["status"] == "error"
        ).sum()) if len(results) else 0,
        "scored_epics": len(scored),
    }])

    return summary, diagnostics


# Ground truth is used here only to validate the experiment population.
# It is never included in the LLM prompt.
preflight = preflight_population()
predictions, llm_calls = run_predictions(preflight)
results = evaluate_predictions(predictions)
summary, diagnostics = evaluation_summary(results, preflight)
llm_call_summary = summarize_llm_calls(llm_calls)

print("\nEvaluation summary")
display(summary)
print("\nPreflight diagnostics")
display(diagnostics)
print("\nLLM latency / token summary")
display(llm_call_summary)
print("\nPer-call LLM metrics")
display(llm_calls.head(50))
if len(results):
    display(results.head(20))

output_path = save_results_excel(
    results,
    EXPERIMENT_NAME,
    "results",
    extra_sheets={
        "evaluation_summary": summary,
        "preflight": preflight,
        "diagnostics": diagnostics,
        "llm_calls": llm_calls,
        "llm_call_summary": llm_call_summary,
    },
)
print(f"Saved {output_path}")
